In [5]:
! python -m pip install --upgrade pip
! pip install tensorflow gensim
! pip install tensorflow gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 87.0 MB/s  0:00:00


In [10]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import gensim

# ==========================================
# 1. LOAD AND PREPROCESS DATA
# ==========================================
print("Loading data...")
data = pd.read_csv('/content/sample_data/IMDB_Dataset.csv', engine='python', on_bad_lines='warn')

# Encode labels
data['sentiment'] = data['sentiment'].map({'positive': 1, 'negative': 0})

# Clean text (remove HTML tags and non-alphanumeric characters)
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text) # Remove HTML
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # Remove punctuation
    return text.lower()

data['review'] = data['review'].apply(clean_text)

# Parameters for text processing
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LENGTH = 200
EMBEDDING_DIM = 100

# Train-Test Split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    data['review'], data['sentiment'], test_size=0.2, random_state=42
)

# Tokenize and Pad Sequences
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

# FIX: Changed padding and truncating to 'pre' for RNN/LSTM/GRU models to work properly
X_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_SEQ_LENGTH, padding='pre', truncating='pre')
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_SEQ_LENGTH, padding='pre', truncating='pre')

# Optimization: Early Stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Function to evaluate and plot metrics
def evaluate_model(model, X_test, y_test, model_name):
    y_prob = model.predict(X_test)
    y_pred = (y_prob > 0.5).astype(int)

    print(f"\n--- {model_name} Evaluation ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"{model_name} - Confusion Matrix")
    plt.savefig(f"{model_name.replace(' ', '_')}_confusion_matrix.png")
    plt.close()

# ==========================================
# 2. MODEL 1: 1D CNN
# ==========================================
print("\nTraining Model 1: 1D CNN...")
model_cnn = Sequential([
    # FIX: Removed the deprecated 'input_length' to stop the warning messages
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5), # Optimization technique
    Dense(1, activation='sigmoid')
])

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, callbacks=[early_stopping])
evaluate_model(model_cnn, X_test, y_test, "1D CNN")

# ==========================================
# 3. MODEL 2: LSTM
# ==========================================
print("\nTraining Model 2: LSTM...")
model_lstm = Sequential([
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM),
    LSTM(64, dropout=0.2), # Dropout applied directly to recurrent layer
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, callbacks=[early_stopping])
evaluate_model(model_lstm, X_test, y_test, "LSTM Model")

# ==========================================
# 4. MODEL 3: GRU WITH GENSIM WORD2VEC & HYPERPARAMETER TUNING
# ==========================================
print("\nTraining Gensim Word2Vec Model...")
# Gensim requires a list of lists (sentences broken into words)
sentences = [text.split() for text in X_train_text]
w2v_model = gensim.models.Word2Vec(sentences, vector_size=EMBEDDING_DIM, window=5, min_count=2, workers=4)

# Create an embedding matrix for the Keras Embedding Layer
word_index = tokenizer.word_index
embedding_matrix = np.zeros((MAX_VOCAB_SIZE, EMBEDDING_DIM))
for word, i in word_index.items():
    if i < MAX_VOCAB_SIZE:
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]

print("\nTraining Model 3: GRU with Word2Vec...")
# HYPERPARAMETER TUNING: Adjusted learning rate and custom Adam optimizer
custom_optimizer = Adam(learning_rate=0.001)
lr_reduction = ReduceLROnPlateau(monitor='val_loss', patience=2, verbose=1, factor=0.5, min_lr=0.0001)

model_gru = Sequential([
    # Pretrained Embeddings used here
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, weights=[embedding_matrix], trainable=True),
    GRU(128, return_sequences=False, dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_gru.compile(optimizer=custom_optimizer, loss='binary_crossentropy', metrics=['accuracy'])
model_gru.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.1, callbacks=[early_stopping, lr_reduction])
evaluate_model(model_gru, X_test, y_test, "GRU with Word2Vec")

Loading data...

Training Model 1: 1D CNN...
Epoch 1/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 77s 154ms/step - accuracy: 0.7781 - loss: 0.4450 - val_accuracy: 0.8734 - val_loss: 0.2938
Epoch 2/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 77s 156ms/step - accuracy: 0.9159 - loss: 0.2238 - val_accuracy: 0.8885 - val_loss: 0.2677
Epoch 3/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 77s 157ms/step - accuracy: 0.9663 - loss: 0.1048 - val_accuracy: 0.8885 - val_loss: 0.3079
Epoch 4/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 74s 150ms/step - accuracy: 0.9915 - loss: 0.0350 - val_accuracy: 0.8968 - val_loss: 0.3529
Epoch 5/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 83s 151ms/step - accuracy: 0.9978 - loss: 0.0121 - val_accuracy: 0.8937 - val_loss: 0.4362
274/274 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step

--- 1D CNN Evaluation ---
Accuracy: 0.8983419096626644

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.90      0.90      4352
           1       0.90      0.89      0.90      4393

    accuracy 